
### Ziel dieser Datei
Berechnungen auf Basis der Grundlagedaten. Hier sollen alle verschnitte/buffer usw. berechnet werden, um überhaupt die Gebiete welche in Frage kommen zu definieren. Am schluss also 1 datei mit allen Flächen die überhaupt in Frage kommen.

### Datenquellen
Die 4 exportierten GeoJSON-Dateien aus Datei 1 einlesen. Es sollte hier keine externen Dateien mehr brauchen.

### Grober Codeaufbau
1. Ausschlusszonen: Bestimmte Neigung ab gewissem Grad oder wenn nicht möglich zumindest die Gefahrenzonen (Klippen) und die Schutzgebiete mit einem Buffer ringsherum komplett ausschliessen. Damit für diese Gebiete gar nicht erst die Möglichkeit besteht, dort ein geeigneter Standort zu zeigen.
2. Ausschlusszonen auschneiden: Die in 1. definierten Zonen sollen mit den infragekommenden Flächen (Wälder/Wiesen) verschnitten werden.
3. Abfrage nach der Infrastruktur: z.B. Puffer um Haltestellen und Bauernhöfe erstellen. Wenn möglich soll dieser Puffer später auch selbst angepasst werden (je nachdem wie wichtig einem dies ist/wie nahe man sein möchte).
4. Alle Lagerflächen, damit sie diesen Kriterien 1-3 entsprechen rausfiltern.

### Export & Übernahme für die Nächste Datei 3
* Finale Liste aller geeigneten Flächen in der Schweiz im GeoJSON-Format mit sinnvoller Bezeichnung.

In [6]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

In [7]:
# 1. Einstellungen

DATA_DIR = Path("data_BL")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

TARGET_CRS = "EPSG:2056"

PATH_FLAECHEN = DATA_DIR / "bearbeitet_flaechen.geojson"
PATH_GEFAHREN = DATA_DIR / "bearbeitet_gefahrenzonen.geojson"
PATH_INFRA = DATA_DIR / "bearbeitet_infrastruktur.geojson"
PATH_HYDRANT = DATA_DIR / "bearbeitet_hydranten.geojson"

# Optional: falls ihr Naturschutzgebiete später noch ergänzt
PATH_NATURSCHUTZ = DATA_DIR / "bearbeitet_schutzgebiete.geojson"

# Parameter
BUFFER_GEFAHREN_M = 30          # Buffer um cliff/scree
BUFFER_NATURSCHUTZ_M = 50       # falls Naturschutzdatei vorhanden ist
MIN_FLAECHE_M2 = 1000           # Mindestfläche für Lagerplatz

In [8]:
# 2. Hilfsfunktionen

def read_layer(path, target_crs=TARGET_CRS):
    if not path.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")
    
    gdf = gpd.read_file(path)
    
    if gdf.crs is None:
        print(f"Achtung: {path.name} hat kein CRS. Es wird {target_crs} angenommen.")
        gdf = gdf.set_crs(target_crs)
    else:
        gdf = gdf.to_crs(target_crs)
    
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf.reset_index(drop=True)
    
    return gdf


def keep_polygons(gdf):
    return gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()


def clean_polygons(gdf):
    if gdf.empty:
        return gdf
    
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    
    # Nur Polygone reparieren
    gdf["geometry"] = gdf.geometry.buffer(0)
    
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    
    return gdf


def make_exclusion_buffer(gdf, buffer_m, typ_name):
    if gdf.empty:
        return gpd.GeoDataFrame(columns=["typ", "geometry"], geometry="geometry", crs=TARGET_CRS)
    
    out = gdf.copy()
    out["geometry"] = out.geometry.buffer(buffer_m)
    out["typ"] = typ_name
    out = out[["typ", "geometry"]]
    out = clean_polygons(out)
    
    return out


def add_nearest_distance(base_gdf, target_gdf, distance_col):
    base_gdf = base_gdf.copy()
    
    if target_gdf.empty:
        base_gdf[distance_col] = np.nan
        return base_gdf
    
    base_tmp = base_gdf.copy()
    base_tmp["_orig_idx"] = base_tmp.index
    
    target_tmp = target_gdf[["geometry"]].copy()
    
    joined = gpd.sjoin_nearest(
        base_tmp,
        target_tmp,
        how="left",
        distance_col=distance_col
    )
    
    distances = joined.groupby("_orig_idx")[distance_col].min()
    base_gdf[distance_col] = base_gdf.index.map(distances)
    
    return base_gdf

In [9]:
# 3. Daten einlesen

flaechen = read_layer(PATH_FLAECHEN)
gefahren = read_layer(PATH_GEFAHREN)
infrastruktur = read_layer(PATH_INFRA)


# Nur Wald und Wiese behalten
flaechen = flaechen[flaechen["landuse"].isin(["forest", "meadow"])].copy()
flaechen = keep_polygons(flaechen)
flaechen = clean_polygons(flaechen)

print("Flächen:", len(flaechen))
print(flaechen["landuse"].value_counts())

print("Gefahrenzonen:", len(gefahren))
print(gefahren["natural"].value_counts())

print("Infrastruktur:", len(infrastruktur))


Flächen: 3470
landuse
meadow    1787
forest    1683
Name: count, dtype: int64
Gefahrenzonen: 592
natural
cliff    586
scree      6
Name: count, dtype: int64
Infrastruktur: 4122


In [10]:
# 4. Infrastruktur aufteilen

# Bauernhöfe
bauernhoefe = infrastruktur[
    infrastruktur["landuse"].fillna("") == "farmyard"
].copy()

# ÖV-Haltestellen: Bushaltestellen und Bahnhöfe
oev = infrastruktur[
    (infrastruktur["highway"].fillna("") == "bus_stop") |
    (infrastruktur["railway"].fillna("") == "station")
].copy()
# Hydranten: aktuell wahrscheinlich leer, weil in deiner Datei kein passendes Attribut vorhanden ist
hydranten = gpd.GeoDataFrame(
    columns=infrastruktur.columns,
    geometry="geometry",
    crs=infrastruktur.crs
)

for col in ["emergency", "amenity", "man_made", "highway"]:
    if col in infrastruktur.columns:
        temp = infrastruktur[
            infrastruktur[col].fillna("").astype(str).str.lower().isin(
                ["fire_hydrant", "hydrant"]
            )
        ].copy()

        hydranten = pd.concat([hydranten, temp], ignore_index=True)

hydranten = gpd.GeoDataFrame(
    hydranten,
    geometry="geometry",
    crs=TARGET_CRS
)

print("Bauernhöfe:", len(bauernhoefe))
print("ÖV-Haltestellen:", len(oev))
print('Hydranten:', len(hydranten))


Bauernhöfe: 676
ÖV-Haltestellen: 1002
Hydranten: 1222


In [11]:
# 5. Ausschlusszonen erstellen

# Geröll und Felsen
gefahren = gefahren[
    gefahren["natural"].isin(["cliff", "scree"])
].copy()

gefahren_buffer = make_exclusion_buffer(
    gefahren,
    BUFFER_GEFAHREN_M,
    "Geröll/Felsen"
)

# Naturschutzgebiete optional
if PATH_NATURSCHUTZ.exists():
    naturschutz = read_layer(PATH_NATURSCHUTZ)
    naturschutz = keep_polygons(naturschutz)
    naturschutz = clean_polygons(naturschutz)
    
    naturschutz_buffer = make_exclusion_buffer(
        naturschutz,
        BUFFER_NATURSCHUTZ_M,
        "Naturschutzgebiet"
    )
else:
    print("Hinweis: Keine Naturschutzgebiete-Datei gefunden.")
    naturschutz_buffer = gpd.GeoDataFrame(
        columns=["typ", "geometry"],
        geometry="geometry",
        crs=TARGET_CRS
    )

# Alles zusammenführen
ausschlusszonen = pd.concat(
    [gefahren_buffer, naturschutz_buffer],
    ignore_index=True
)

ausschlusszonen = gpd.GeoDataFrame(
    ausschlusszonen,
    geometry="geometry",
    crs=TARGET_CRS
)

ausschlusszonen = clean_polygons(ausschlusszonen)

print("Ausschlusszonen:", len(ausschlusszonen))

Ausschlusszonen: 37197


In [12]:
# 7. Ausschlusszonen aus Wald/Wiese ausschneiden

if not ausschlusszonen.empty:
    ausschluss_union = gpd.GeoDataFrame(
        geometry=[ausschlusszonen.geometry.unary_union],
        crs=TARGET_CRS
    )
    
    lagerflaechen = gpd.overlay(
        flaechen,
        ausschluss_union,
        how="difference",
        keep_geom_type=True
    )
else:
    lagerflaechen = flaechen.copy()

lagerflaechen = clean_polygons(lagerflaechen)

print("Lagerflächen nach Ausschluss:", len(lagerflaechen))

C:\Users\elias\AppData\Local\Temp\ipykernel_27184\3473452451.py:5: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[ausschlusszonen.geometry.unary_union],


Lagerflächen nach Ausschluss: 2824


In [13]:
# 8. Kleine Restflächen entfernen

lagerflaechen["flaeche_m2"] = lagerflaechen.geometry.area
lagerflaechen["flaeche_ha"] = lagerflaechen["flaeche_m2"] / 10000

lagerflaechen = lagerflaechen[
    lagerflaechen["flaeche_m2"] >= MIN_FLAECHE_M2
].copy()

lagerflaechen = lagerflaechen.reset_index(drop=True)

print("Lagerflächen nach Mindestfläche:", len(lagerflaechen))
print("Gesamtfläche in ha:", round(lagerflaechen["flaeche_ha"].sum(), 2))

Lagerflächen nach Mindestfläche: 2012
Gesamtfläche in ha: 40918.12


In [14]:
# 9. Distanz zu Bauernhof, Hydrant und ÖV berechnen

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    bauernhoefe,
    "dist_bauernhof_m"
)

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    hydranten,
    "dist_hydrant_m"
)

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    oev,
    "dist_oev_m"
)

lagerflaechen[[
    "flaeche_ha",
    "dist_bauernhof_m",
    "dist_hydrant_m",
    "dist_oev_m"
]].head()

,flaeche_ha,dist_bauernhof_m,dist_hydrant_m,dist_oev_m
0,1542.534783,0.000000,198.871803,0.000000
1,2.526824,1831.236680,4550.031334,2599.119605
2,793.498344,0.000000,288.672900,282.784720
3,83.241120,5.148998,1657.691388,260.603980
4,111.526647,5.319830,1873.461513,234.301773


In [15]:
# 10. Finale Lagerflächen vorbereiten

lagerflaechen_final = lagerflaechen.copy()
lagerflaechen_final = lagerflaechen_final.reset_index(drop=True)

lagerflaechen_final["lagerplatz_id"] = range(1, len(lagerflaechen_final) + 1)

print("Finale Lagerflächen:", len(lagerflaechen_final))

Finale Lagerflächen: 2012


In [16]:
# 12. Unnötige Spalten reduzieren

spalten_behalten = [
    "lagerplatz_id",
    "landuse",
    "flaeche_m2",
    "flaeche_ha",
    "dist_bauernhof_m",
    "dist_hydrant_m",
    "dist_oev_m",
    "geometry"
]

lagerflaechen_final = lagerflaechen_final[
    [col for col in spalten_behalten if col in lagerflaechen_final.columns]
].copy()

In [17]:
# 13. Export für Notebook 3

output_path = OUT_DIR / "geeignete_lagerflaechen_BL.geojson"

lagerflaechen_final.to_file(
    output_path,
    driver="GeoJSON"
)

print("Export abgeschlossen:")
print(output_path)

Export abgeschlossen:
output\geeignete_lagerflaechen_BL.geojson
